In [1]:
import pandas as pd
import numpy as np
from churn_prediction.paths import PROCESSED_ONLINE_RETAIL_DIR, FEATURES_ONLINE_RETAIL_DIR
import warnings
warnings.filterwarnings('ignore')

All directories created successfully!


In [2]:
df = pd.read_parquet(PROCESSED_ONLINE_RETAIL_DIR / 'online_retail_clean.parquet')
df['Revenue'] = df['Quantity'] * df['Price']

print(f"Data range : {df['InvoiceDate'].min().date()} → {df['InvoiceDate'].max().date()}")
print(f"Customers  : {df['Customer ID'].nunique():,}")
print(f"Total rows : {len(df):,}")

Data range : 2009-12-01 → 2011-12-09
Customers  : 5,878
Total rows : 779,425


## Chiến lược tạo nhãn
    Label:
    churn = 1  →  khách KHÔNG mua gì trong 90 ngày sau snap_date
    churn = 0  →  khách CÓ ít nhất 1 giao dịch trong 90 ngày sau snap_date
**Rolling window**: Tạo snapshot mỗi tháng để tăng số lượng samples

In [3]:
CHURN_WINDOW_DAYS   = 90   # không mua trong 90 ngày → churn
OBS_MIN_DAYS        = 90   # khách phải có ít nhất 90 ngày lịch sử mới được tính
SNAPSHOT_FREQ       = 'MS' # tạo snapshot mỗi đầu tháng
# Tạo snapshot
data_start = df['InvoiceDate'].min()
data_end   = df['InvoiceDate'].max()

# Snapshot phải đủ chỗ cho cả observation lẫn churn window
snap_start = data_start + pd.Timedelta(days=OBS_MIN_DAYS)
snap_end   = data_end   - pd.Timedelta(days=CHURN_WINDOW_DAYS)

snapshots = pd.date_range(start=snap_start, end=snap_end, freq=SNAPSHOT_FREQ)
print(f"Snapshots  : {len(snapshots)} điểm ({snapshots[0].date()} → {snapshots[-1].date()})")

Snapshots  : 19 điểm (2010-03-01 → 2011-09-01)


In [4]:
records = []

for snap in snapshots:
    churn_end = snap + pd.Timedelta(days=CHURN_WINDOW_DAYS)

    # Khách có giao dịch TRƯỚC snap (có lịch sử để tính features)
    customers_obs = (
        df[df['InvoiceDate'] < snap]['Customer ID']
        .unique()
    )

    # Khách có giao dịch TRONG cửa sổ churn
    customers_active = set(
        df[
            (df['InvoiceDate'] >= snap) &
            (df['InvoiceDate'] <  churn_end)
        ]['Customer ID'].unique()
    )

    for cid in customers_obs:
        records.append({
            'Customer ID'   : cid,
            'snapshot_date' : snap,
            'churn'         : 0 if cid in customers_active else 1,
        })

labels = pd.DataFrame(records)

print(f"Total samples  : {len(labels):,}")
print(f"Unique customers: {labels['Customer ID'].nunique():,}")
print(f"\nChurn rate overall:")
print(labels['churn'].value_counts(normalize=True).rename({0:'retained', 1:'churned'}))

Total samples  : 73,439
Unique customers: 5,249

Churn rate overall:
churn
churned     0.586391
retained    0.413609
Name: proportion, dtype: float64


In [5]:
# Kiểm tra churn rate theo từng snapshot — nếu bất thường thì cần xem lại
churn_by_snap = (
    labels
    .groupby('snapshot_date')['churn']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'churn_rate', 'count': 'n_customers'})
)
print(churn_by_snap.to_string())

                     churn_rate  n_customers
snapshot_date                               
2010-03-01 07:45:00    0.382009         1712
2010-04-01 07:45:00    0.432947         2155
2010-05-01 07:45:00    0.468763         2449
2010-06-01 07:45:00    0.504624         2703
2010-07-01 07:45:00    0.513286         2973
2010-08-01 07:45:00    0.458056         3159
2010-09-01 07:45:00    0.419151         3321
2010-10-01 07:45:00    0.469136         3564
2010-11-01 07:45:00    0.558995         3941
2010-12-01 07:45:00    0.669245         4266
2011-01-01 07:45:00    0.677107         4342
2011-02-01 07:45:00    0.666893         4413
2011-03-01 07:45:00    0.650209         4537
2011-04-01 07:45:00    0.649067         4716
2011-05-01 07:45:00    0.646827         4822
2011-06-01 07:45:00    0.671599         4933
2011-07-01 07:45:00    0.659591         5041
2011-08-01 07:45:00    0.624344         5143
2011-09-01 07:45:00    0.573062         5249


In [6]:
labels.to_parquet(
    FEATURES_ONLINE_RETAIL_DIR / 'labels.parquet',
    index=False
)
print(f"Saved labels.parquet: {labels.shape}")
labels.head(10)

Saved labels.parquet: (73439, 3)


,Customer ID,snapshot_date,churn
0,13085.0,2010-03-01 07:45:00,1
1,13078.0,2010-03-01 07:45:00,0
2,15362.0,2010-03-01 07:45:00,1
3,18102.0,2010-03-01 07:45:00,0
4,12682.0,2010-03-01 07:45:00,0
5,18087.0,2010-03-01 07:45:00,0
6,13635.0,2010-03-01 07:45:00,1
7,14110.0,2010-03-01 07:45:00,0
8,12636.0,2010-03-01 07:45:00,1
9,17519.0,2010-03-01 07:45:00,0
